In [1]:
import os
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from keras.applications.mobilenet_v2 import MobileNetV2


# Load data
directory = "data/Indian-monuments/images/train"
base_path = os.path.abspath(directory)
class_folders = os.listdir(base_path)

images = []
labels = []
class_counts = {}

for class_name in class_folders:
    class_path = os.path.join(base_path, class_name)
    for image_file in os.listdir(class_path):
        try:
            image_path = os.path.join(class_path, image_file)
            # Check if the file is a valid image file
            if not os.path.isfile(image_path) or not image_path.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue  # Skip non-image files
            image = tf.keras.preprocessing.image.load_img(image_path, target_size=(299, 299))  # InceptionV3 input size
            image = tf.keras.preprocessing.image.img_to_array(image)
            images.append(image)
            labels.append(class_name)
            class_counts[class_name] = class_counts.get(class_name, 0) + 1
        except Exception as e:
            print(f"Error loading {image_path}: {str(e)}")

X = np.array(images)
y = labels

# Preprocess labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split the data into training, validation, and testing sets
X_train, X_val_test, y_train_encoded, y_val_test_encoded = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y)
X_test, X_val, y_test_encoded, y_val_encoded = train_test_split(X_val_test, y_val_test_encoded, test_size=0.5, random_state=42)

from keras.models import Model
from keras.layers import GlobalAveragePooling2D, Dense

# Preprocess input data
X_train_preprocessed = tf.keras.applications.mobilenet_v2.preprocess_input(X_train)
X_val_preprocessed = tf.keras.applications.mobilenet_v2.preprocess_input(X_val)
X_test_preprocessed = tf.keras.applications.mobilenet_v2.preprocess_input(X_test)

# Load MobileNetV2 base model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(299, 299, 3))

# Add custom classification head
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(512, activation='relu')(x)
predictions = Dense(len(class_folders), activation='softmax')(x)

# Combine base model and custom classification head
model = Model(inputs=base_model.input, outputs=predictions)

# Freeze base model layers
for layer in base_model.layers:
    layer.trainable = False

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model.fit(X_train_preprocessed, y_train_encoded, 
                    validation_data=(X_val_preprocessed, y_val_encoded), 
                    epochs=10, batch_size=32)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test_preprocessed, y_test_encoded)
print(f'Test accuracy: {test_accuracy}')




9406464/9406464 [==============================] - 40s 4us/step
Epoch 1/10
92/92 [==============================] - 83s 869ms/step - loss: 0.8862 - accuracy: 0.7725 - val_loss: 0.3314 - val_accuracy: 0.8910
Epoch 2/10
92/92 [==============================] - 87s 951ms/step - loss: 0.1720 - accuracy: 0.9546 - val_loss: 0.2416 - val_accuracy: 0.9319
Epoch 3/10
92/92 [==============================] - 133s 1s/step - loss: 0.0878 - accuracy: 0.9795 - val_loss: 0.1821 - val_accuracy: 0.9482
Epoch 4/10
92/92 [==============================] - 80s 872ms/step - loss: 0.0398 - accuracy: 0.9939 - val_loss: 0.1675 - val_accuracy: 0.9591
Epoch 5/10
92/92 [==============================] - 87s 950ms/step - loss: 0.0258 - accuracy: 0.9952 - val_loss: 0.1572 - val_accuracy: 0.9700
Epoch 6/10
92/92 [==============================] - 87s 949ms/step - loss: 0.0177 - accuracy: 0.9986 - val_loss: 0.1483 - val_accuracy: 0.9755
Epoch 7/10
92/92 [==============================] - 80s 874ms/step - loss: 0.010